In [3]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [7, 5]

In [4]:
# availability_file = './Study/data_availability_whereCAMorAirgo.csv'
availability_file = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/study/data_availability.csv'
data_partitions_id = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/study/data_partitions_id.csv'

In [5]:
availability_data = pd.read_csv(availability_file)
availability_data['category'] = -1
availability_data.iloc[:10]

,study_id,date,day_night,cam_icu,airgo,bedmaster,category
0,1,2018-06-05,day,False,False,True,-1
1,1,2018-06-05,night,False,False,True,-1
2,1,2018-06-06,day,False,True,True,-1
3,1,2018-06-06,night,False,True,True,-1
4,1,2018-06-07,day,False,False,True,-1
5,1,2018-06-07,night,False,False,True,-1
6,1,2018-06-08,day,False,False,True,-1
7,2,2018-06-11,day,False,True,True,-1
8,2,2018-06-11,night,False,True,True,-1
9,2,2018-06-12,day,False,True,True,-1


In [6]:
# create ID for each night and day. id format: d01, n01, d02, n02, d03,...

In [7]:
availability_data['id'] = -1

In [8]:
availability_data.date = pd.to_datetime(availability_data.date, infer_datetime_format=1)

In [9]:
for study_id in pd.unique(availability_data.study_id):
    
    days_diff = [x.days for x in (availability_data.loc[np.logical_and(availability_data.study_id.values == study_id,availability_data.day_night.values == 'day')].date - availability_data.loc[availability_data.study_id.values == study_id].date.iloc[0])]
    nights_diff = [x.days for x in (availability_data.loc[np.logical_and(availability_data.study_id.values == study_id,availability_data.day_night.values == 'night')].date - availability_data.loc[availability_data.study_id.values == study_id].date.iloc[0])]
        
    availability_data.loc[np.logical_and(availability_data.study_id.values == study_id,availability_data.day_night.values == 'day'),'id'] = [str(study_id).zfill(3) + '_d'+str(i+1).zfill(2) for i in days_diff]
    availability_data.loc[np.logical_and(availability_data.study_id.values == study_id,availability_data.day_night.values == 'night'),'id'] = [str(study_id).zfill(3) + '_n'+str(i+1).zfill(2) for i in nights_diff]


In [10]:
availability_data.date = availability_data.date.apply(lambda x: str(x.date()))

In [11]:
availability_data.loc[np.logical_and(availability_data.airgo.values, availability_data.bedmaster.values),'category'] = 'A'
availability_data.loc[np.logical_and(availability_data.airgo.values, 1- availability_data.bedmaster.values),'category'] = 'B'
availability_data.loc[np.logical_and(1-availability_data.airgo.values, availability_data.bedmaster.values),'category'] = 'C'

In [15]:
availability_data.to_csv(data_partitions_id, index=False)

In [16]:
categories_n, counts_n = np.unique(availability_data.category[availability_data.day_night=='night'], return_counts=1)
print('Nights:')
print(categories_n)
print(counts_n)
categories_d, counts_d = np.unique(availability_data.category[availability_data.day_night=='day'], return_counts=1)
print('Days:')
print(categories_d)
print(counts_d)

Nights:
['A' 'B' 'C']
[275  71 555]
Days:
['A' 'B' 'C']
[375  90 489]


In [26]:
night_counts = pd.DataFrame({'category':categories_n, 'nights':counts_n, 'days':counts_d})
night_counts.set_index('category',inplace=True)
ax = night_counts.plot.bar(rot=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+0.05, p.get_height() + 2))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [18]:
nightA = availability_data[np.logical_and(availability_data.day_night=='night', availability_data.category=='A')].copy()
nightB = availability_data[np.logical_and(availability_data.day_night=='night', availability_data.category=='B')].copy()
nightAB = availability_data[np.logical_and(availability_data.day_night=='night', np.logical_or(availability_data.category=='A', availability_data.category=='B'))].copy()
nightC = availability_data[np.logical_and(availability_data.day_night=='night', availability_data.category=='C')].copy()

In [19]:
study_ids_A, counts_A = np.unique(nightA.study_id, return_counts=True)
study_ids_B, counts_B = np.unique(nightB.study_id, return_counts=True)
study_ids_C, counts_C = np.unique(nightC.study_id, return_counts=True)
study_ids_AB, counts_AB = np.unique(nightAB.study_id, return_counts=True)

print('number of patients with at least one night with category A: ')
print(study_ids_A.shape[0])
print('number of patients with at least one night with category A or B: ')
print(study_ids_AB.shape[0])

number of patients with at least one night with category A: 
138
number of patients with at least one night with category A or B: 
149


In [20]:
# non A B studyIDs:
print(list(set(range(1,190)) - set(study_ids_AB)))

[132, 5, 134, 136, 9, 138, 141, 144, 147, 148, 22, 23, 150, 26, 27, 28, 159, 162, 166, 40, 41, 170, 174, 55, 184, 57, 58, 59, 60, 62, 65, 66, 70, 73, 75, 94, 96, 99, 124, 127]


In [21]:
airgo_available = pd.DataFrame()
airgo_available['airgo_available'] = [str(x).zfill(3) for x in study_ids_AB]

In [22]:
airgo_available.to_csv('icusleep_airgoavailable.csv', index=False)

In [23]:
# ax = plt.subplot(121)
ax = pd.DataFrame(counts_A).plot.hist(bins=np.arange(np.max(counts_A))+0.5, rwidth =0.9, legend=False)  #bins=(np.max(counts_A)-1)*2)
ax.set_title('Distribution of category A nights per subject')
ax.set_xlabel('Category A nights per subject')
ax.set_ylabel('Amount of subjects')
for p in ax.patches:
    if p.get_height()<2: continue
    ax.annotate(str(int(p.get_height())), (p.get_x() + 0.35, p.get_height() + 0.7))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [24]:
ax2 = pd.DataFrame(counts_AB).plot.hist(bins=np.arange(np.max(counts_AB))+0.5, rwidth =0.9, legend=False)  #bins=(np.max(counts_A)-1)*2)
ax2.set_title('Distribution of category A and B nights per subject')
ax2.set_xlabel('Category A and B nights per subject')
ax2.set_ylabel('Amount of subjects')
for p in ax2.patches:
    if p.get_height()<2: continue
    ax2.annotate(str(int(p.get_height())), (p.get_x() + 0.1, p.get_height() + 0.7))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [25]:
nightAB.to_csv('nights_AB.csv', index=False)